# Influencer Recommendation System — Clean Data Pipeline

**MongoDB (raw) → Pandas cleaning/features → Supabase PostgreSQL**

Run top-to-bottom. The PostgreSQL load uses **full refresh (`replace`)**, so rerunning does not append duplicate rows. This drops and recreates the two destination tables; do not use if other systems depend on their existing schema/permissions without checking first.


## 0. Install dependencies

In [19]:
%pip install --no-cache-dir --upgrade cryptography pyOpenSSL pymongo dnspython

   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   -- ------------------------------------- 0.3/3.8 MB ? eta -:--:--
   ----- ---------------------------------- 0.5/3.8 MB 1.1 MB/s eta 0:00:03
   ---------- ----------------------------- 1.0/3.8 MB 1.5 MB/s eta 0:00:02
   ------------- -------------------------- 1.3/3.8 MB 1.4 MB/s eta 0:00:02
   ---------------- ----------------------- 1.6/3.8 MB 1.3 MB/s eta 0:00:02
   ------------------- -------------------- 1.8/3.8 MB 1.5 MB/s eta 0:00:02
   ------------------- -------------------- 1.8/3.8 MB 1.5 MB/s eta 0:00:02
   ------------------- -------------------- 1.8/3.8 MB 1.5 MB/s eta 0:00:02
   ------------------- -------------------- 1.8/3.8 MB 1.5 MB/s eta 0:00:02
   ------------------------ --------------- 2.4

In [18]:
%pip uninstall -y pyOpenSSL cryptography


Found existing installation: pyOpenSSL 25.0.0
Uninstalling pyOpenSSL-25.0.0:
  Successfully uninstalled pyOpenSSL-25.0.0
Found existing installation: cryptography 46.0.5
Uninstalling cryptography-46.0.5:
  Successfully uninstalled cryptography-46.0.5
Note: you may need to restart the kernel to use updated packages.


You can safely remove it manually.


In [1]:
%pip install -q pymongo pandas sqlalchemy psycopg2-binary sentence-transformers python-dotenv




Note: you may need to restart the kernel to use updated packages.


## 1. Configure credentials safely

**Do not paste credentials into notebook code or commit them to Git.** Enter each connection URI when prompted. For Supabase, copy the URI from **Project → Connect → Session pooler** (or your selected connection method). Paste the complete URI as provided; don't manually add backslashes. If credentials previously appeared in a shared notebook/chat, rotate them first.


In [2]:
import os
from getpass import getpass

import numpy as np
import pandas as pd

import pymongo,cryptography
from bson import ObjectId
from sqlalchemy import create_engine, text
from sqlalchemy.engine import make_url

In [8]:


MONGO_URI = os.getenv("MONGO_URI") or getpass("MongoDB URI: ")
POSTGRES_URI = os.getenv("POSTGRES_URI") or getpass("Supabase PostgreSQL URI: ")

DB_NAME = "influencer_db"
ML_SCHEMA = "ml_data"
MIN_VIDEOS = 5
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

if not MONGO_URI.strip() or not POSTGRES_URI.strip():
    raise ValueError("Both MongoDB and PostgreSQL connection URIs are required.")

# Validate PostgreSQL URI without printing its password.
pg_url = make_url(POSTGRES_URI)
if pg_url.get_backend_name() != "postgresql":
    raise ValueError("POSTGRES_URI must be a PostgreSQL URI.")
print("Configuration looks valid; credentials are not displayed.")


Configuration looks valid; credentials are not displayed.


## 2. Extract MongoDB data

In [4]:
import sys
import pymongo
import ssl

print("Python:", sys.executable)
print("PyMongo:", pymongo.version)
print("OpenSSL:", ssl.OPENSSL_VERSION)

Python: c:\Users\bhada\anaconda3\python.exe
PyMongo: 4.16.0
OpenSSL: OpenSSL 3.0.16 11 Feb 2025


In [9]:
def load_raw_data(mongo_uri: str, db_name: str = DB_NAME):
    client = pymongo.MongoClient(
        mongo_uri,
        serverSelectionTimeoutMS=15000
    )

    client.admin.command("ping")

    db = client[db_name]

    channels = pd.DataFrame(list(db["channels"].find({})))
    videos = pd.DataFrame(list(db["videos"].find({})))

    print(f"Loaded {len(channels):,} channels and {len(videos):,} videos")

    return client, db, channels, videos


mongo_client, db, channel_df, videos_df = load_raw_data(MONGO_URI)

C:\Users\bhada\AppData\Roaming\Python\Python313\site-packages\pymongo\pyopenssl_context.py:348: CryptographyDeprecationWarning: Parsed a serial number which wasn't positive (i.e., it was negative or zero), which is disallowed by RFC 5280. Loading this certificate will cause an exception in a future release of cryptography.
  _crypto.X509.from_cryptography(x509.load_der_x509_certificate(cert))


Loaded 1,849 channels and 17,770 videos


In [10]:
def load_raw_data(mongo_uri: str, db_name: str = DB_NAME):
    client = pymongo.MongoClient(mongo_uri, serverSelectionTimeoutMS=15000)
    client.admin.command("ping")
    db = client[db_name]
    channels = pd.DataFrame(list(db["channels"].find({})))
    videos = pd.DataFrame(list(db["videos"].find({})))
    print(f"Loaded {len(channels):,} channels and {len(videos):,} videos")
    return client, db, channels, videos

mongo_client, db, channel_df, videos_df = load_raw_data(MONGO_URI)

Loaded 1,849 channels and 17,770 videos


## 3. Flatten nested MongoDB fields (safe to rerun)

In [11]:
def flatten_nested_column(df: pd.DataFrame, column: str) -> pd.DataFrame:
    df = df.copy()
    if column not in df.columns:
        return df
    nested = pd.json_normalize(df[column].map(lambda x: x if isinstance(x, dict) else {}))
    # Keep existing top-level values if a flattened key already exists.
    nested = nested[[c for c in nested.columns if c not in df.columns]]
    df = pd.concat([df.drop(columns=[column]).reset_index(drop=True),
                    nested.reset_index(drop=True)], axis=1)
    return df

videos_df = flatten_nested_column(videos_df, "statistics")
channel_df = flatten_nested_column(channel_df, "statistics")

if "thumbnails" in channel_df.columns:
    channel_df["thumbnail_url"] = channel_df["thumbnails"].map(
        lambda x: x.get("default", {}).get("url") if isinstance(x, dict) else None
    )
    channel_df = channel_df.drop(columns=["thumbnails"])

print("Flattened:", channel_df.shape, videos_df.shape)


Flattened: (1849, 23) (17770, 17)


## 4. Clean and reconcile channel/video records

In [12]:
def clean_pipeline(channel_df: pd.DataFrame, videos_df: pd.DataFrame,
                  min_videos: int = MIN_VIDEOS):
    channels = channel_df.copy()
    videos = videos_df.copy()

    debug_cols = [
        "search_keyword", "video_ingestion_error",
        "video_ingestion_last_attempt"
    ]
    channels = channels.drop(columns=debug_cols, errors="ignore")

    for col in ["category", "location_type", "country"]:
        if col in channels.columns:
            channels[col] = channels[col].fillna("Unknown")

    # Only apply ingestion-status filtering if that field exists.
    if "video_ingestion_status" in channels.columns:
        channels = channels[channels["video_ingestion_status"].notna()].copy()

    if "channel_id" not in channels.columns or "channel_id" not in videos.columns:
        raise KeyError("Both collections must contain 'channel_id'.")

    counts = videos.groupby("channel_id").size().rename("actual_video_count")
    channels = channels.drop(columns=["actual_video_count"], errors="ignore")
    channels = channels.merge(counts, on="channel_id", how="left")
    channels["actual_video_count"] = channels["actual_video_count"].fillna(0).astype("int64")
    channels = channels[channels["actual_video_count"] >= min_videos].copy()

    valid_ids = set(channels["channel_id"].dropna())
    videos = videos[videos["channel_id"].isin(valid_ids)].copy()

    print(f"Cleaned to {len(channels):,} channels and {len(videos):,} videos")
    return channels.reset_index(drop=True), videos.reset_index(drop=True)

channel_df, videos_df = clean_pipeline(channel_df, videos_df)


Cleaned to 954 channels and 17,770 videos


## 5. Feature engineering

In [13]:
def numeric_column(df: pd.DataFrame, name: str) -> pd.Series:
    if name not in df.columns:
        return pd.Series(0, index=df.index, dtype="float64")
    return pd.to_numeric(df[name], errors="coerce").fillna(0)

def engineer_video_features(videos: pd.DataFrame) -> pd.DataFrame:
    videos = videos.copy()
    # YouTube API uses viewCount/likeCount/commentCount; some exports use snake_case.
    aliases = {
        "view_count": ["view_count", "viewCount", "views"],
        "like_count": ["like_count", "likeCount", "likes"],
        "comment_count": ["comment_count", "commentCount", "comments"],
    }
    for target, candidates in aliases.items():
        source = next((c for c in candidates if c in videos.columns), None)
        videos[target] = pd.to_numeric(videos[source], errors="coerce").fillna(0) if source else 0.0

    denom = videos["view_count"].where(videos["view_count"] > 0)
    videos["engagement_rate"] = (
        (videos["like_count"] + videos["comment_count"]) / denom
    ).replace([np.inf, -np.inf], np.nan).fillna(0.0)
    return videos

def engineer_channel_features(channels: pd.DataFrame, videos: pd.DataFrame) -> pd.DataFrame:
    channels = channels.copy()
    channels = channels.drop(
        columns=["avg_engagement_rate", "description_embedding"] +
                [c for c in channels.columns if c.startswith("category_")],
        errors="ignore"
    )
    agg = videos.groupby("channel_id")["engagement_rate"].mean().rename(
        "avg_engagement_rate"
    )
    channels = channels.merge(agg, on="channel_id", how="left")
    channels["avg_engagement_rate"] = channels["avg_engagement_rate"].fillna(0.0)

    # One-hot categories, consistent with this dataset's observed categories.
    if "category" in channels.columns:
        dummies = pd.get_dummies(channels["category"].fillna("Unknown"), prefix="category", dtype=bool)
        channels = pd.concat([channels.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
    return channels

videos_df = engineer_video_features(videos_df)
channel_df = engineer_channel_features(channel_df, videos_df)
display(channel_df.head(3))
display(videos_df.head(3))


,_id,channel_id,country,description,last_updated,published_at,source,title,uploads_playlist_id,video_ingested_at,...,category_beauty,category_fashion,category_finance,category_fitness,category_food,category_gaming,category_health,category_lifestyle,category_parenting,category_travel
0,6a76259a28c9961fa402924c,UCJ2e_MJaqdPmcf-FiMrZdIA,US,"¡Hola, Mi Bella! ¿Lista para transformar tu cu...",2026-08-07 18:36:12.787,2017-11-07T17:07:08Z,youtube,Cardio Fit by India Fitness,UUJ2e_MJaqdPmcf-FiMrZdIA,2026-08-07 19:03:08.093,...,False,False,False,False,False,False,False,False,False,False
1,6a76259b28c9961fa402924d,UCjiEkwIdaueJLz2y327pQtw,IN,Fitness Rockers is India's fastest growing Fit...,2026-08-07 18:36:13.139,2015-01-07T07:39:40Z,youtube,FitnessRockers India,UUjiEkwIdaueJLz2y327pQtw,2026-08-07 19:03:10.514,...,False,False,False,False,False,False,False,False,False,False
2,6a76259b28c9961fa402924e,UCdRUxAdk6aLASXlb64cLOJg,IN,hello Guys Welcome Back On World Best Bodybuil...,2026-08-07 18:36:13.158,2021-06-16T06:44:07.592155Z,youtube,Antic Fitness India,UUdRUxAdk6aLASXlb64cLOJg,2026-08-07 19:03:14.450,...,False,False,False,False,False,False,False,False,False,False


,_id,video_id,category,category_id,channel_id,channel_title,description,duration,last_updated,location_type,published_at,source,tags,title,view_count,like_count,comment_count,engagement_rate
0,6a762be928c9961fa402963c,0CliP431wQc,None,17,UCJ2e_MJaqdPmcf-FiMrZdIA,Cardio Fit by India Fitness,,PT32M30S,2026-08-07 19:03:07.256,None,2026-08-03T23:39:35Z,youtube,[],Cardio Fit by India Fitness está en vivo,273,28,1,0.106227
1,6a762be928c9961fa402963d,P86RfQN_YHo,None,17,UCJ2e_MJaqdPmcf-FiMrZdIA,Cardio Fit by India Fitness,,PT35M6S,2026-08-07 19:03:07.313,None,2026-07-27T23:43:37Z,youtube,[],Cardio Fit full body,1761,34,1,0.019875
2,6a762be928c9961fa402963e,nHL544w2xaw,None,17,UCJ2e_MJaqdPmcf-FiMrZdIA,Cardio Fit by India Fitness,,PT28S,2026-08-07 19:03:07.338,None,2026-07-23T21:03:49Z,youtube,[],"Rutina de abdomen, glúteos y piernas #rutina ...",3782,106,8,0.030143


## 6. Optional text embeddings

Embedding every channel description can take time and memory. Run this only if your downstream model needs embeddings. The output is stored as a comma-separated string in PostgreSQL; for production vector search, consider `pgvector`.


In [14]:
from sentence_transformers import SentenceTransformer

def add_description_embeddings(channels: pd.DataFrame,
                               model_name: str = EMBED_MODEL_NAME) -> pd.DataFrame:
    result = channels.copy()
    if "description" not in result.columns:
        print("No description column; skipping embeddings.")
        return result
    model = SentenceTransformer(model_name)
    vectors = model.encode(
        result["description"].fillna("").astype(str).tolist(),
        show_progress_bar=True,
        convert_to_numpy=True
    )
    result["description_embedding"] = [v.tolist() for v in vectors]
    return result

# Uncomment if required (may take time):
# channel_df = add_description_embeddings(channel_df)


## 7. Prepare PostgreSQL-compatible values

In [15]:
def postgres_safe_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    def convert(value):
        if value is None or value is pd.NA:
            return None
        if isinstance(value, ObjectId):
            return str(value)
        if isinstance(value, np.ndarray):
            return ",".join(map(str, value.tolist()))
        if isinstance(value, (list, tuple)):
            # Preserve scalar lists as a simple text representation.
            return ",".join(map(str, value))
        if isinstance(value, dict):
            return str(value)
        if isinstance(value, np.generic):
            return value.item()
        if isinstance(value, float) and not np.isfinite(value):
            return None
        return value

    for col in out.columns:
        if out[col].dtype == "object":
            out[col] = out[col].map(convert)
        elif pd.api.types.is_datetime64_any_dtype(out[col]):
            out[col] = pd.to_datetime(out[col], errors="coerce")
    # SQL column names must be unique.
    if not out.columns.is_unique:
        raise ValueError("Duplicate column names found; resolve them before upload.")
    return out

pg_channels = postgres_safe_frame(channel_df)
pg_videos = postgres_safe_frame(videos_df)

print("Ready for SQL:", pg_channels.shape, pg_videos.shape)


Ready for SQL: (954, 33) (17770, 18)


## 8. Load into Supabase PostgreSQL (rerunnable full refresh)

This replaces the two tables on each run rather than appending. It avoids duplicates from repeated notebook execution. Run only when a full refresh is intended.


In [17]:
def load_to_postgres(channels: pd.DataFrame, videos: pd.DataFrame,
                     postgres_uri: str, schema: str = ML_SCHEMA):
    if not schema.replace("_", "").isalnum():
        raise ValueError("Schema name may contain only letters, numbers, and underscores.")
    if channels.empty and videos.empty:
        raise ValueError("Both input DataFrames are empty; refusing to replace tables.")

    safe_channels = postgres_safe_frame(channels)
    safe_videos = postgres_safe_frame(videos)

    engine = create_engine(postgres_uri, pool_pre_ping=True)
    try:
        with engine.begin() as conn:
            conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{schema}"'))

        # Full refresh: rerunning replaces rows, not appends duplicates.
        safe_channels.to_sql(
            "channels", engine, schema=schema, if_exists="replace",
            index=False, chunksize=500, method="multi"
        )
        safe_videos.to_sql(
            "videos", engine, schema=schema, if_exists="replace",
            index=False, chunksize=500, method="multi"
        )
        print(f"Upload complete: {len(safe_channels):,} channels, "
              f"{len(safe_videos):,} videos → {schema}")
    finally:
        engine.dispose()

# Run after checking the preview above:
load_to_postgres(channel_df, videos_df, POSTGRES_URI, schema=ML_SCHEMA)


Upload complete: 954 channels, 17,770 videos → ml_data


## 9. One-call end-to-end run

In [ ]:
def run_pipeline(mongo_uri: str, postgres_uri: str,
                 min_videos: int = MIN_VIDEOS,
                 schema: str = ML_SCHEMA,
                 add_embeddings: bool = False):
    client, db, channels, videos = load_raw_data(mongo_uri)
    try:
        videos = flatten_nested_column(videos, "statistics")
        channels = flatten_nested_column(channels, "statistics")
        if "thumbnails" in channels.columns:
            channels["thumbnail_url"] = channels["thumbnails"].map(
                lambda x: x.get("default", {}).get("url") if isinstance(x, dict) else None
            )
            channels = channels.drop(columns=["thumbnails"])

        channels, videos = clean_pipeline(channels, videos, min_videos)
        videos = engineer_video_features(videos)
        channels = engineer_channel_features(channels, videos)
        if add_embeddings:
            channels = add_description_embeddings(channels)

        load_to_postgres(channels, videos, postgres_uri, schema=schema)
        return channels, videos
    finally:
        client.close()

# Explicitly uncomment to execute the complete pipeline:
# channel_df, videos_df = run_pipeline(MONGO_URI, POSTGRES_URI)


## Notes / operating guidance

- This notebook is appropriate for **exploration, cleaning experiments, visual checks, and model development**.
- Move stable functions into `.py` modules when you need scheduled/repeated execution, testing, code review, or use from an API/backend. Keep the notebook as a thin experiment/runner.
- This version does not write reconciled counts back to MongoDB; cleaning is read-only.
- `replace` makes reruns duplicate-safe but recreates table definitions. If you need incremental updates and preserved constraints, use primary keys plus PostgreSQL `INSERT ... ON CONFLICT` upserts instead.
- The notebook cannot verify your live credentials or Supabase schema until you run it in your environment.
